# Rent vs Income Project Dataset Builder
This notebook pulls Census (ACS) data and prepares it for merging with Zillow rent data.

In [5]:
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

In [6]:
#Get Census Data
API_KEY = "275f7b174beaf7cc3e2b1a64e8992a92b3be1643"
BASE_URL = "https://api.census.gov/data/2022/acs/acs5" 

In [11]:
import pandas as pd
import requests

# DMV ZIP LIST
zip_list = [
    '20001','20002','20003','20004','20005','20006','20007','20008','20009','20010',
    '20011','20012','20015','20016','20017','20018','20019','20020','20024','20032',
    '20036','20037','20105','20109','20110','20111','20120','20121','20132','20136',
    '20147','20148','20151','20152','20155','20164','20165','20166','20169','20170',
    '20171','20175','20176','20186','20187','20190','20191','20194','20526','20601',
    '20603','20613','20657','20678','20705','20706','20707','20708','20710','20712',
    '20715','20716','20721','20737','20740','20743','20744','20745','20746','20747',
    '20748','20770','20772','20774','20781','20782','20783','20784','20785','20814',
    '20815','20816','20817','20850','20851','20852','20853','20854','20855','20866',
    '20871','20874','20876','20877','20878','20879','20886','20895','20901','20902',
    '20903','20904','20906','20910','20912','21701','21702','21703','21704','21774',
    '21793','22003','22015','22026','22030','22031','22033','22041','22042','22043',
    '22044','22046','22060','22079','22101','22102','22150','22151','22152','22153',
    '22180','22181','22182','22191','22192','22193','22201','22202','22203','22204',
    '22205','22206','22207','22209','22301','22302','22303','22304','22305','22306',
    '22307','22309','22310','22311','22312','22314','22315','22401','22405','22406',
    '22407','22408','22553','22554','22556','22630','22701','25414','25438']

#VARIABLES
variables = [
    "NAME",
    "B19013_001E",  # income
    "B25064_001E",  # rent
    "B25003_001E",  # occupied
    "B25003_003E",  # renter
    "B17001_001E",
    "B17001_002E",
    "B01003_001E",
    "B15003_022E","B15003_023E","B15003_024E","B15003_025E",
    "B03002_001E","B03002_003E","B03002_004E","B03002_006E","B03002_012E",
    "B01001_002E","B01001_026E",
    "B01001_003E","B01001_004E","B01001_005E","B01001_006E",
    "B01001_027E","B01001_028E","B01001_029E","B01001_030E",
    "B01001_020E","B01001_021E","B01001_022E","B01001_023E",
    "B01001_024E","B01001_025E",
    "B01001_044E","B01001_045E","B01001_046E","B01001_047E",
    "B01001_048E","B01001_049E"]

#LOOP OVER YEARS
years = [2018, 2019, 2020, 2021, 2022]
all_data = []

for year in years:
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    
    params = {
        "get": ",".join(variables),
        "for": "zip code tabulation area:*",
        "key": API_KEY}
    
    response = requests.get(url, params=params)
    data = response.json()
    
    temp = pd.DataFrame(data[1:], columns=data[0])
    temp["year"] = year
    all_data.append(temp)

df = pd.concat(all_data, ignore_index=True)
df = df.rename(columns={
    "zip code tabulation area": "zip",
    "B19013_001E": "median_income",
    "B25064_001E": "median_rent",
    "B25003_001E": "occupied_units",
    "B25003_003E": "renter_units",
    "B17001_001E": "poverty_universe",
    "B17001_002E": "below_poverty",
    "B01003_001E": "population",
    "B15003_022E": "bachelors",
    "B15003_023E": "masters",
    "B15003_024E": "professional",
    "B15003_025E": "doctorate",
    "B03002_001E": "race_total",
    "B03002_003E": "white",
    "B03002_004E": "black",
    "B03002_006E": "asian",
    "B03002_012E": "hispanic",
    "B01001_002E": "male",
    "B01001_026E": "female"
})

#Numeric conversion
for col in df.columns:
    if col not in ["NAME", "zip"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

#Fix zip formatting
df["zip"] = df["zip"].astype(str).str.zfill(5)

# FILTER TO DMV
df = df[df["zip"].isin(zip_list)]

# FEATURE ENGINEERING
df["monthly_income"] = df["median_income"] / 12
df["rent_to_income"] = df["median_rent"] / df["monthly_income"]

df["pct_renter"] = df["renter_units"] / df["occupied_units"]
df["poverty_rate"] = df["below_poverty"] / df["poverty_universe"]

df["bachelors_plus"] = df["bachelors"] + df["masters"] + df["professional"] + df["doctorate"]
df["pct_bachelors_plus"] = df["bachelors_plus"] / df["population"]

# race
df["pct_white"] = df["white"] / df["race_total"]
df["pct_black"] = df["black"] / df["race_total"]
df["pct_asian"] = df["asian"] / df["race_total"]
df["pct_hispanic"] = df["hispanic"] / df["race_total"]

# gender
df["pct_male"] = df["male"] / df["population"]
df["pct_female"] = df["female"] / df["population"]

# age
under18_cols = [
    "B01001_003E","B01001_004E","B01001_005E","B01001_006E",
    "B01001_027E","B01001_028E","B01001_029E","B01001_030E"]

elderly_cols = [
    "B01001_020E","B01001_021E","B01001_022E","B01001_023E",
    "B01001_024E","B01001_025E",
    "B01001_044E","B01001_045E","B01001_046E","B01001_047E",
    "B01001_048E","B01001_049E"]

df["pct_under18"] = df[under18_cols].sum(axis=1) / df["population"]
df["pct_elderly"] = df[elderly_cols].sum(axis=1) / df["population"]

#FINAL DATASET
final_df = df[[
    "zip","year",
    "median_income","median_rent","rent_to_income",
    "pct_renter","poverty_rate","pct_bachelors_plus",
    "pct_white","pct_black","pct_asian","pct_hispanic",
    "pct_male","pct_female",
    "pct_under18","pct_elderly"]]

final_df.head()

KeyboardInterrupt: 

## Use Zillow Data to get Zip code list

In [10]:
#read csv as pd df
zillow_df = pd.read_csv(DATA_DIR / "Zip_zori.csv")

#Filter to DMV metro
zillow_df = zillow_df[
    zillow_df["Metro"] == "Washington-Arlington-Alexandria, DC-VA-MD-WV"]

zips = zillow_df["RegionName"].astype(str).str.zfill(5).unique()
zip_list = sorted(zips.tolist())
zip_list

['20001',
 '20002',
 '20003',
 '20004',
 '20005',
 '20006',
 '20007',
 '20008',
 '20009',
 '20010',
 '20011',
 '20012',
 '20015',
 '20016',
 '20017',
 '20018',
 '20019',
 '20020',
 '20024',
 '20032',
 '20036',
 '20037',
 '20105',
 '20109',
 '20110',
 '20111',
 '20120',
 '20121',
 '20132',
 '20136',
 '20147',
 '20148',
 '20151',
 '20152',
 '20155',
 '20164',
 '20165',
 '20166',
 '20169',
 '20170',
 '20171',
 '20175',
 '20176',
 '20186',
 '20187',
 '20190',
 '20191',
 '20194',
 '20526',
 '20601',
 '20603',
 '20613',
 '20657',
 '20678',
 '20705',
 '20706',
 '20707',
 '20708',
 '20710',
 '20712',
 '20715',
 '20716',
 '20721',
 '20737',
 '20740',
 '20743',
 '20744',
 '20745',
 '20746',
 '20747',
 '20748',
 '20770',
 '20772',
 '20774',
 '20781',
 '20782',
 '20783',
 '20784',
 '20785',
 '20814',
 '20815',
 '20816',
 '20817',
 '20850',
 '20851',
 '20852',
 '20853',
 '20854',
 '20855',
 '20866',
 '20871',
 '20874',
 '20876',
 '20877',
 '20878',
 '20879',
 '20886',
 '20895',
 '20901',
 '20902',
